# Baseline 7 — SHAP-LightGBM·버전별 LSTM 통합 실험

`baseline_5_howto.ipynb`의 정상 IDW 공간집계·램프 피처·SHAP 선별 LightGBM과
`baseline_6.ipynb`의 LSTM v1/v2/v3를 **동일한 전처리·시간순 검증 구간**에서 비교한다.
검증 총점이 가장 높은 제출 가능 모델을 KPX 그룹별로 다시 전체 학습해
`submission_baseline7.csv`까지 생성한다.

| Step | 내용 |
|------|------|
| 0 | 환경 설정 — 라이브러리·경로·시드 |
| 1 | 정상 IDW + 예보 변화량 피처 전처리 및 캐시 |
| 2 | LightGBM 전체/SHAP 선별 피처 학습 |
| 3 | 동일 선별 피처로 LSTM v1/v2/v3 공정 비교 |
| 4 | 공식 총점 집계 + 그룹별 최적 모델 선택 |
| 5 | 전체 학습 + 평가 기간 예측 + 제출 검증 |

### 통합 원칙

- `turbine_meta["group"]`은 `"kpx_group_N"`으로 맞추고 IDW 신호가 0이 아닌지 즉시 검사한다.
- 모든 모델은 같은 마지막 20% 검증 구간과 `SEQ_LEN=24` 이후 시각으로 평가한다.
- SHAP 선별은 학습 구간으로 학습한 모델과 검증 표본만 사용하며, LSTM의 결측 대체·스케일러도 학습 구간에만 적합한다.
- Persistence는 직전 실제값을 쓰는 oracle 참고선이므로 최종 모델 선택에서 제외한다.
- baseline 6에서 이미 확인한 4개 피처 조합을 다시 36회 반복하지 않고, baseline 5 HOWTO의 선별 피처를 세 LSTM에 공통 적용해 모델 구조 효과만 비교한다.


---
## Step 0. 환경 설정

LightGBM·SHAP·PyTorch가 없는 환경에서만 다음 설치 줄의 주석을 해제해 한 번 실행한다.
설치 후에는 커널을 재시작한다.


In [ ]:
# %pip install lightgbm shap torch scikit-learn
# 현재 커널에 패키지가 없을 때만 실행한다.


In [ ]:
import sys
from pathlib import Path

EXPECTED_VENV = "WINDFORCE/.venv"
# 이 노트북이 돌아야 하는 가상환경 경로 조각
if EXPECTED_VENV not in Path(sys.executable).as_posix():
    raise RuntimeError(
        f"잘못된 커널입니다: {sys.executable}\n"
        f"VS Code 우상단 커널 선택에서 WINDFORCE/.venv를 고르세요.\n"
        f"주피터랩이면 kernel.json의 argv[0]이 절대경로인지 확인하세요."
    )
    # 전역 파이썬으로 뜬 커널을 서드파티 임포트 전에 차단한다.
print(f"kernel: {sys.executable}")
# 어느 인터프리터로 돌았는지 실행 기록에 남긴다.

import os
import re
import time
import warnings
from typing import Optional

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from sklearn.preprocessing import MinMaxScaler
# 데이터 분석·트리 모델·설명가능성·딥러닝 라이브러리

warnings.filterwarnings("ignore")
# 반복 실험 로그를 가리는 라이브러리 버전 경고를 억제한다.


In [ ]:
ROOT: str = next(
    (
        path
        for path in [
            "D:/workspaces/WINDFORCE",
            "d:/workspaces/WINDFORCE",
            "/Users/ksydata/WINDFORCE",
        ]
        if os.path.exists(path)
    ),
    os.getcwd(),
)
# Windows/macOS 실행 환경에서 실제로 존재하는 프로젝트 루트를 선택한다.
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Windforce 패키지 탐색 경로를 추가한다.

from Windforce import (
    EvaluationMetrics,
    GFSFeatureEngineer,
    LDAPSFeatureEngineer,
    RATED_CAPACITY_KW,
    TIME_STEP_HOURS,
    WindforceDataLoader,
)
# 데이터 로더·전처리기·공식 평가지표를 루트 패키지에서 가져온다.
from Windforce.Modeling import (
    AllFeaturesVariant,
    BaselineModels,
    WindforceDatasetBuilder,
    make_lstm_pipeline,
)
# baseline 5 HOWTO와 baseline 6에서 검증한 모델링 구성요소를 재사용한다.

print(f"ROOT: {ROOT}")
print(f"lightgbm {lgb.__version__} / shap {shap.__version__} / torch {torch.__version__}")
# 실행 경로와 핵심 라이브러리 버전을 재현 로그에 남긴다.


In [ ]:
pd.set_option("display.max_columns", None)
# 데이터프레임의 모든 컬럼을 생략 없이 표시한다.
pd.options.display.float_format = "{:,.6f}".format
# 지수 표기 대신 소수점 6자리 실수로 표시한다.

GROUPS = [1, 2, 3]
# KPX 평가 그룹 번호 목록
SEQ_LEN = 24
# 최근 24시간의 기상 흐름으로 다음 시각 발전량(kWh)을 예측한다.
SEED = 42
# 모든 확률적 모델과 표본 추출에 쓰는 재현 시드
TEST_RATIO = 0.2
# 시간순 마지막 20%를 공통 검증 구간으로 사용한다.
SHAP_SAMPLE_SIZE = 2_000
# SHAP 계산 속도를 위한 그룹별 검증 표본 수
CORR_THRESHOLD = 0.95
# 이 값을 넘는 절대 상관 피처 쌍에서 SHAP 순위가 낮은 쪽을 제거한다.
NOISE_FEATURE_COUNT = 3
# null importance 기준선을 만들 무작위 정규분포 피처 개수
CUMULATIVE_RATIO = 1.0
# baseline 5 HOWTO 실측 최적값으로, 노이즈·상관 제거 뒤 꼬리 절단은 하지 않는다.
PREP_DIR = f"{ROOT}/prep"
# baseline 7 전처리 완료본을 저장하는 폴더

os.makedirs(PREP_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(8)
# 12코어 전부를 쓰면 배치 64 구간에서 스레드 동기화 비용이 연산량을 넘어선다.
# numpy와 PyTorch 난수 시드를 고정한다.


---
## Step 1. 정상 IDW + 예보 변화량 피처 전처리 및 캐시

```text
LDAPS 16격자 / GFS 9격자
            ↓ 물리 파생변수
turbine_meta(group 문자열·위경도·용량) 기반 IDW
            ↓
KPX 그룹 × 시각 공간가중 피처
            ↓ transformForecastFeature()
1h/3h/6h 변화량·램프 피처
            ↓
prep/baseline7_dataset_{train|test}_groupN.csv.gz
```

캐시는 baseline 7 전용 파일명을 써서 이전 노트북의 구버전 전처리 결과와 섞이지 않게 한다.


In [ ]:
def parseDMS(coord_str: str) -> tuple[float, float]:
    """도분초(DMS) 좌표를 십진수(DD) 위도·경도로 변환하는 함수

    Args:
        - coord_str: `37°16'55.61"N 128°57'02.10"E` 형식 문자열

    Returns:
        - (위도, 경도) 십진수 튜플 (도 단위)
    """
    tokens = re.findall(r"[\d.]+|[NSEW]", coord_str.replace('""', '"'))
    # 숫자와 방향(N/S/E/W)을 순서대로 분리한다.
    if len(tokens) != 8:
        raise ValueError(f"지원하지 않는 DMS 좌표 형식입니다: {coord_str}")
    # 위도·경도 각각 도/분/초/방향 4개 토큰인지 검증한다.

    lat = float(tokens[0]) + float(tokens[1]) / 60.0 + float(tokens[2]) / 3_600.0
    lon = float(tokens[4]) + float(tokens[5]) / 60.0 + float(tokens[6]) / 3_600.0
    # 도 + 분/60 + 초/3,600으로 십진도 단위로 환산한다.
    if tokens[3] == "S":
        lat = -lat
    if tokens[7] == "W":
        lon = -lon
    # 남반구(S)와 서경(W)은 음수 좌표로 변환한다.
    return lat, lon
    # IDW 거리 계산에 사용할 (위도, 경도) 튜플을 반환한다.


def buildTurbineMeta(kpx_info: pd.DataFrame) -> pd.DataFrame:
    """KPX 터빈 메타를 IDW 공간집계 계약에 맞게 변환하는 함수

    Args:
        - kpx_info: 터빈 1기 = 1행인 원본 메타 DataFrame

    Returns:
        - group / lat / lon / cap_kw 컬럼을 가진 DataFrame
    """
    kpx = kpx_info.copy()
    # 호출자가 보관한 원본 DataFrame을 변형하지 않는다.
    coords = kpx["좌표(Google)"].apply(parseDMS)
    kpx["lat"] = coords.apply(lambda value: value[0])
    kpx["lon"] = coords.apply(lambda value: value[1])
    # DMS 좌표를 십진 위도·경도(도 단위)로 분리한다.

    meta = kpx[["KPX그룹", "lat", "lon", "설비용량(MW)"]].rename(
        columns={"KPX그룹": "group", "설비용량(MW)": "cap_kw"}
    ).copy()
    meta["cap_kw"] = meta["cap_kw"] * 1_000.0
    # 설비용량을 MW에서 kW로 즉시 환산한다.
    meta["group"] = "kpx_group_" + meta["group"].astype(int).astype(str)
    # IDW 공통 계약과 같은 문자열 키로 맞춰 0 가중치 행렬 재발을 차단한다.
    return meta.reset_index(drop=True)
    # 행렬 인덱스와 터빈 행 순서를 일치시켜 반환한다.


In [ ]:
class PreprocessedDatasetBuilder:
    """LDAPS·GFS 원본을 baseline 7 그룹별 학습·평가 캐시로 만드는 클래스

    사용 순서:
        builder = PreprocessedDatasetBuilder(loader, PREP_DIR)
        builder.buildAll()
        train_df = builder.load(1, "train")
    """

    def __init__(
        self,
        loader: WindforceDataLoader,
        prep_dir: str,
        groups: list[int] = GROUPS,
    ):
        self.loader = loader
        # 원본 CSV를 지연 로드·캐시하는 데이터 로더
        self.prep_dir = prep_dir
        # gzip 전처리 완료본 저장 폴더
        self.groups = groups
        # 처리할 KPX 그룹 번호 목록
        self.turbine_meta: Optional[pd.DataFrame] = None
        # buildAll()에서 생성하는 IDW용 터빈 메타

    def _path(self, group: int, split: str) -> str:
        """그룹·데이터 구간별 baseline 7 캐시 경로를 만드는 내부 메서드"""
        return f"{self.prep_dir}/baseline7_dataset_{split}_group{group}.csv.gz"
        # pyarrow 추가 의존성 없이 압축 가능한 gzip CSV 경로를 반환한다.

    def isCached(self) -> bool:
        """모든 그룹의 학습·평가 캐시가 존재하는지 확인하는 메서드"""
        return all(
            os.path.exists(self._path(group, split))
            for group in self.groups
            for split in ("train", "test")
        )
        # 파일 하나라도 빠지면 전체 train/test를 같은 코드 버전으로 다시 만든다.

    def _transformSplit(
        self,
        ldaps_key: str,
        gfs_key: str,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """LDAPS·GFS 한 구간을 그룹 단위 예보 피처로 변환하는 내부 메서드"""
        ldaps_fe = LDAPSFeatureEngineer()
        gfs_fe = GFSFeatureEngineer()
        # 전체 격자를 처리한 뒤 같은 터빈 메타로 세 그룹에 공간 매핑한다.

        ldaps_grid = ldaps_fe.transformLDAPS(self.loader[ldaps_key])
        ldaps_group = ldaps_fe.transformToGroupIDW(ldaps_grid, self.turbine_meta)
        ldaps_group = ldaps_fe.transformForecastFeature(ldaps_group)
        # LDAPS 물리 피처 → IDW → 그룹별 1h/3h/6h 변화량 순서로 처리한다.

        gfs_grid = gfs_fe.transformGFS(self.loader[gfs_key])
        gfs_group = gfs_fe.transformToGroupIDW(gfs_grid, self.turbine_meta)
        gfs_group = gfs_fe.transformForecastFeature(gfs_group)
        # GFS 물리 피처 → IDW → 허브풍속 기반 변화량 순서로 처리한다.

        print(f"  LDAPS: 격자 {ldaps_grid.shape} → 그룹 {ldaps_group.shape}")
        print(f"  GFS  : 격자 {gfs_grid.shape} → 그룹 {gfs_group.shape}")
        # 원본 격자와 그룹 집계 shape를 실행 로그에 남긴다.
        return ldaps_group, gfs_group
        # 한 행 = 한 그룹 × 한 시각인 두 예보표를 반환한다.

    @staticmethod
    def _checkSignal(ldaps_group: pd.DataFrame, gfs_group: pd.DataFrame) -> None:
        """IDW 결과의 대표 풍속 신호가 0 행렬이 아닌지 검사하는 내부 메서드"""
        checks = {
            "LDAPS ws10": (ldaps_group, "ws10"),
            "GFS gfs_ws_hub": (gfs_group, "gfs_ws_hub"),
        }
        # 각 기상원에서 발전량과 직접 연결되는 대표 풍속(m/s)을 검사한다.
        for name, (df, column) in checks.items():
            if column not in df.columns or float(df[column].abs().max()) <= 0.0:
                raise ValueError(f"{name} IDW 신호가 없거나 전부 0입니다")
            # 그룹 키나 공간가중 버그가 조용히 학습 단계로 넘어가지 않게 즉시 실패한다.

    def buildAll(self, force: bool = False) -> None:
        """전체 그룹의 baseline 7 학습·평가 캐시를 생성하는 메서드

        Args:
            - force: True면 기존 baseline 7 캐시를 무시하고 다시 만든다.
        """
        if self.isCached() and not force:
            print("✅ baseline 7 전처리 캐시가 있어 재사용합니다.")
            return
            # 같은 접두사의 6개 캐시가 모두 있으면 약 1분의 전처리를 건너뛴다.

        started = time.time()
        self.turbine_meta = buildTurbineMeta(self.loader["kpx_info"])
        # 정규화된 그룹 키·위경도·설비용량(kW)을 한 번만 만든다.

        print("[train] 격자 전처리 + IDW 그룹 집계 중...")
        ldaps_train, gfs_train = self._transformSplit("ldaps_train", "gfs_train")
        print("[test] 격자 전처리 + IDW 그룹 집계 중...")
        ldaps_test, gfs_test = self._transformSplit("ldaps_test", "gfs_test")
        self._checkSignal(ldaps_train, gfs_train)
        self._checkSignal(ldaps_test, gfs_test)
        # 학습·평가 양쪽에서 대표 풍속 피처가 실제 값인지 확인한다.

        train_builder = WindforceDatasetBuilder(
            ldaps_train,
            gfs_train,
            self.loader["train_labels"],
        )
        # 학습 예보 피처에 그룹별 실제 발전량(kWh)을 시각 기준으로 결합한다.

        test_times = pd.to_datetime(pd.Series(ldaps_test["forecast_kst_dtm"].unique()))
        dummy_labels = pd.DataFrame({"kst_dtm": test_times})
        for group in self.groups:
            dummy_labels[f"kpx_group_{group}"] = 0.0
        # 평가 기간은 라벨이 없으므로 같은 빌더 경로를 쓰기 위한 0 kWh 더미를 만든다.
        test_builder = WindforceDatasetBuilder(ldaps_test, gfs_test, dummy_labels)
        # 학습·평가 피처 병합 규칙과 컬럼 접미사를 완전히 동일하게 유지한다.

        for group in self.groups:
            for split, builder in (("train", train_builder), ("test", test_builder)):
                table = builder.build(group)
                path = self._path(group, split)
                table.to_csv(
                    path,
                    index=False,
                    encoding="utf-8-sig",
                    compression="gzip",
                )
                # UTF-8 BOM·gzip 형식으로 저장해 한글 호환성과 용량을 함께 지킨다.
                print(f"✅ [그룹{group}] {split}: {table.shape} → {os.path.basename(path)}")
        print(f"전처리 완료: {time.time() - started:.1f}초")
        # 전체 캐시 생성 시간을 재현 로그에 남긴다.

    def load(self, group: int, split: str = "train") -> pd.DataFrame:
        """저장한 그룹 캐시를 읽어 시간순으로 반환하는 메서드"""
        path = self._path(group, split)
        if not os.path.exists(path):
            raise FileNotFoundError(f"baseline 7 전처리 캐시가 없습니다: {path}")
        # buildAll() 실행 누락을 명확한 파일 경로와 함께 알린다.
        df = pd.read_csv(path, encoding="utf-8-sig")
        df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
        # CSV 왕복에서 문자열이 된 예보 대상 시각을 datetime으로 복원한다.
        return df.sort_values("forecast_kst_dtm").reset_index(drop=True)
        # 시계열 분할과 제출 재색인이 안전하도록 정렬 상태로 반환한다.


In [ ]:
# 데이터 경로를 확인하고 baseline 7 전처리 캐시를 준비한다.
loader = WindforceDataLoader(root=ROOT)
path_status = loader.check_paths()
if not all(path_status.values()):
    missing = [name for name, exists in path_status.items() if not exists]
    raise FileNotFoundError(f"필수 입력 파일이 없습니다: {missing}")
# 일부 파일이 없는 상태로 긴 전처리를 시작하지 않게 막는다.

prep_builder = PreprocessedDatasetBuilder(loader, PREP_DIR)
prep_builder.buildAll()
# 캐시가 없을 때만 정상 IDW·램프 피처 전처리를 실행한다.


In [ ]:
def splitByTime(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    test_ratio: float = TEST_RATIO,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    """그룹 데이터를 학습·검증 구간으로 시간순 분할하는 함수

    Returns:
        - X_train, X_valid, y_train, y_valid 튜플
    """
    split = int(len(df) * (1.0 - test_ratio))
    # 앞 80%를 학습, 뒤 20%를 미래 검증 구간으로 고정한다.
    X_train = df.iloc[:split][feature_cols].reset_index(drop=True)
    X_valid = df.iloc[split:][feature_cols].reset_index(drop=True)
    y_train = df.iloc[:split][target_col].to_numpy(dtype=float)
    y_valid = df.iloc[split:][target_col].to_numpy(dtype=float)
    # 피처와 발전량(kWh) 타깃을 같은 행 경계로 잘라 인덱스를 맞춘다.
    return X_train, X_valid, y_train, y_valid
    # 랜덤 셔플 없이 시간 순서를 보존한 네 객체를 반환한다.


datasets: dict[int, dict] = {}
# 그룹별 전체 표·공통 피처·시간순 분할 결과를 보관한다.
for group in GROUPS:
    df_group = prep_builder.load(group, "train")
    target_col = f"kpx_group_{group}"
    feature_cols = WindforceDatasetBuilder.featureCols(df_group)
    X_train, X_valid, y_train, y_valid = splitByTime(
        df_group,
        target_col,
        feature_cols,
    )
    datasets[group] = {
        "df": df_group,
        "feature_cols": feature_cols,
        "X_train": X_train,
        "X_valid": X_valid,
        "y_train": y_train,
        "y_valid": y_valid,
    }
    # 모든 모델이 같은 행·피처·검증 구간을 재사용하도록 한곳에 고정한다.
    print(
        f"[그룹{group}] 전체 {df_group.shape} | 피처 {len(feature_cols)}개 | "
        f"train {len(X_train):,} / valid {len(X_valid):,}"
    )
    # 그룹별 데이터 shape와 공통 분할 크기를 확인한다.


---
## Step 2. LightGBM 전체/SHAP 선별 피처 학습

먼저 127개 안팎의 전체 피처로 LightGBM을 학습한다. 검증 표본의 평균 절대 SHAP 기여도와
무작위 노이즈 기준선을 계산하고, 노이즈 이하 피처 및 절대 상관 0.95 초과 중복 피처를 제거한다.
같은 하이퍼파라미터로 선별 피처 모델을 다시 학습해 피처 선택의 순수 효과를 비교한다.

모든 모델 지표는 LSTM과 같은 시각을 쓰기 위해 검증 구간의 첫 `SEQ_LEN=24`시간을 제외한다.


In [ ]:
LGB_PARAMS = {
    "objective": "regression_l1",
    # NMAE가 절대오차 기반이므로 학습 목적을 L1로 맞춘다.
    "n_estimators": 1_200,
    # 실제 트리 수는 검증 L1 조기종료가 결정한다.
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 40,
    # 파워커브 비선형을 표현하되 작은 노이즈 구간의 과적합을 억제한다.
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    # 행·열 샘플링과 L2 정규화로 트리 간 상관을 낮춘다.
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1,
}
# 전체/선별 모델에 같은 설정을 적용해 피처 선택 효과만 비교한다.

metrics = EvaluationMetrics(
    rated_capacity_kw=RATED_CAPACITY_KW,
    time_step_hours=TIME_STEP_HOURS,
)
# KPX 그룹별 설비용량(kW)과 1시간 해상도를 쓰는 공식 지표 계산기


def trainLightGBM(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    capacity_kw: float,
) -> lgb.LGBMRegressor:
    """설비용량 대비 이용률로 LightGBM을 학습하는 함수"""
    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_train,
        y_train / capacity_kw,
        # 발전량(kWh)을 설비용량(kW) 대비 0~1 이용률로 정규화한다.
        eval_X=X_valid,
        eval_y=y_valid / capacity_kw,
        eval_metric="l1",
        callbacks=[
            lgb.early_stopping(80, verbose=False),
            lgb.log_evaluation(0),
        ],
        # 검증 L1이 80개 트리 동안 개선되지 않으면 최적 반복에서 멈춘다.
    )
    return model
    # best_iteration_이 기록된 학습 모델을 반환한다.


def predictKw(
    model: lgb.LGBMRegressor,
    X: pd.DataFrame,
    capacity_kw: float,
) -> np.ndarray:
    """LightGBM 이용률 예측을 발전량(kWh)으로 환산하는 함수"""
    utilization = np.clip(model.predict(X), 0.0, 1.0)
    # 물리적으로 가능한 설비 이용률 0~1 범위로 제한한다.
    return utilization * capacity_kw
    # 1시간 자료이므로 이용률 × 설비용량(kW) = 시간당 발전량(kWh)이다.


def summarizeAll(
    pred: np.ndarray,
    actual: np.ndarray,
    group: int,
) -> dict:
    """공식 지표와 MAE·RMSE(kWh)를 함께 반환하는 함수"""
    summary = metrics.summarize(pred, actual, group)
    # NMAE·FICR·그룹 총점을 대회 정의 그대로 계산한다.
    summary["mae"] = float(np.mean(np.abs(pred - actual)))
    summary["rmse"] = float(np.sqrt(np.mean((pred - actual) ** 2)))
    # 직관적인 오차 규모를 시간당 발전량(kWh) 단위로 추가한다.
    return summary
    # 리더보드 입력 스키마로 합칠 지표 딕셔너리를 반환한다.


In [ ]:
class FeatureSelector:
    """null importance와 상관 중복으로 SHAP 피처를 선별하는 클래스

    사용 순서:
        selector = FeatureSelector(capacity_kw=RATED_CAPACITY_KW[1])
        columns = selector.select(X_train, y_train, X_valid, y_valid, shap_importance)
    """

    def __init__(
        self,
        capacity_kw: float,
        corr_threshold: float = CORR_THRESHOLD,
        noise_count: int = NOISE_FEATURE_COUNT,
        cumulative_ratio: float = CUMULATIVE_RATIO,
        seed: int = SEED,
    ):
        self.capacity_kw = capacity_kw
        # 이용률 학습에 쓰는 그룹 설비용량(kW)
        self.corr_threshold = corr_threshold
        # 절대 상관 중복 판정 임계값
        self.noise_count = noise_count
        # null importance 기준선용 무작위 피처 개수
        self.cumulative_ratio = cumulative_ratio
        # 상위 SHAP 누적 기여 유지 비율
        self.seed = seed
        # 무작위 노이즈 재현 시드
        self.noise_threshold_: Optional[float] = None
        self.report_: Optional[pd.DataFrame] = None
        # select() 실행 뒤 채워지는 노이즈 기준과 피처별 판정표

    @staticmethod
    def _shapArray(model: lgb.LGBMRegressor, sample: pd.DataFrame) -> np.ndarray:
        """SHAP 버전별 반환 형식을 2차원 배열로 통일하는 내부 메서드"""
        values = shap.TreeExplainer(model).shap_values(sample)
        # 트리 구조를 직접 순회해 표본 × 피처 SHAP 기여도를 계산한다.
        if isinstance(values, list):
            values = values[0]
        # 일부 SHAP 버전이 회귀에서도 list를 반환하는 경우 첫 출력을 사용한다.
        return np.asarray(values)
        # shape = (표본 수, 피처 수)인 numpy 배열로 반환한다.

    def _computeNoiseImportance(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_valid: pd.DataFrame,
        y_valid: np.ndarray,
        sample_index: pd.Index,
    ) -> pd.Series:
        """노이즈 피처를 포함한 모델의 평균 절대 SHAP 기여도를 계산하는 내부 메서드"""
        rng = np.random.default_rng(self.seed)
        X_train_noise = X_train.copy()
        X_valid_noise = X_valid.copy()
        # 호출자가 보관한 공통 분할 DataFrame을 변형하지 않는다.
        for index in range(self.noise_count):
            X_train_noise[f"__noise_{index}"] = rng.normal(size=len(X_train_noise))
            X_valid_noise[f"__noise_{index}"] = rng.normal(size=len(X_valid_noise))
            # 타깃과 무관한 표준정규 피처를 학습·검증 양쪽에 독립 생성한다.

        model = trainLightGBM(
            X_train_noise,
            y_train,
            X_valid_noise,
            y_valid,
            self.capacity_kw,
        )
        sample = X_valid_noise.loc[sample_index]
        values = self._shapArray(model, sample)
        importance = pd.Series(
            np.abs(values).mean(axis=0),
            index=X_train_noise.columns,
        )
        # 방향이 상쇄되지 않도록 표본 평균 |SHAP|을 피처 중요도로 사용한다.
        noise_cols = [f"__noise_{index}" for index in range(self.noise_count)]
        self.noise_threshold_ = float(importance[noise_cols].max())
        # 노이즈 중 최대 기여도를 우연히 얻을 수 있는 보수적 하한으로 정한다.
        return importance
        # 원래 피처와 노이즈 피처의 같은 척도 중요도를 반환한다.

    def select(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_valid: pd.DataFrame,
        y_valid: np.ndarray,
        shap_importance: pd.Series,
        sample_index: pd.Index,
    ) -> list[str]:
        """노이즈·상관·누적 기여 기준으로 피처를 선별하는 메서드"""
        feature_cols = X_train.columns.tolist()
        # 원래 학습 피처 순서를 판정표 작성에 보존한다.
        noise_importance = self._computeNoiseImportance(
            X_train,
            y_train,
            X_valid,
            y_valid,
            sample_index,
        )
        survived = [
            column
            for column in feature_cols
            if noise_importance.get(column, 0.0) > self.noise_threshold_
        ]
        dropped_noise = set(feature_cols) - set(survived)
        # 노이즈 최대 기여도 이하인 피처를 정보 없는 후보로 제거한다.

        corr = X_train[survived].corr().abs()
        # 검증 통계를 보지 않고 학습 구간의 절대 상관만 계산한다.
        ordered = [column for column in shap_importance.index if column in survived]
        selected: list[str] = []
        dropped_corr: dict[str, str] = {}
        for column in ordered:
            duplicate = next(
                (
                    kept
                    for kept in selected
                    if corr.loc[column, kept] > self.corr_threshold
                ),
                None,
            )
            if duplicate is None:
                selected.append(column)
            else:
                dropped_corr[column] = duplicate
            # SHAP 상위 피처를 먼저 남기고 같은 정보를 가진 하위 피처를 제거한다.

        ranked = shap_importance.loc[selected]
        if self.cumulative_ratio < 1.0:
            cumulative = ranked.cumsum() / ranked.sum()
            keep_count = int((cumulative < self.cumulative_ratio).sum()) + 1
            selected = ranked.index[:keep_count].tolist()
        # 1.0이면 baseline 5 HOWTO 실측대로 꼬리 절단을 적용하지 않는다.

        rows = []
        for column in feature_cols:
            if column in dropped_noise:
                decision = "노이즈 이하"
            elif column in dropped_corr:
                decision = f"중복({dropped_corr[column]})"
            elif column in selected:
                decision = "선택"
            else:
                decision = "누적 기여 꼬리"
            rows.append(
                {
                    "feature": column,
                    "shap_importance": float(shap_importance.get(column, 0.0)),
                    "decision": decision,
                }
            )
            # 피처별 제거·선택 사유를 재현 가능한 표로 기록한다.
        self.report_ = pd.DataFrame(rows).sort_values(
            "shap_importance",
            ascending=False,
        ).reset_index(drop=True)
        # 사람이 상위 피처와 제거 사유를 함께 검토할 수 있게 정렬한다.
        return selected
        # 평균 |SHAP| 내림차순의 최종 피처 목록을 반환한다.


In [ ]:
records: list[dict] = []
# 그룹·모델별 공통 지표 행을 누적한다.
lgb_models: dict[tuple[int, str], lgb.LGBMRegressor] = {}
selected_features: dict[int, list[str]] = {}
selection_reports: dict[int, pd.DataFrame] = {}
shap_store: dict[int, tuple[np.ndarray, pd.DataFrame, pd.Series]] = {}
# 최종 학습과 해석에 재사용할 모델·피처·SHAP 산출물을 보관한다.

for group in GROUPS:
    data = datasets[group]
    X_train = data["X_train"]
    X_valid = data["X_valid"]
    y_train = data["y_train"]
    y_valid = data["y_valid"]
    capacity_kw = RATED_CAPACITY_KW[group]
    actual_aligned = y_valid[SEQ_LEN:]
    # 모든 모델을 LSTM과 같은 검증 시각으로 비교하기 위해 앞 24시간을 제외한다.

    model_full = trainLightGBM(
        X_train,
        y_train,
        X_valid,
        y_valid,
        capacity_kw,
    )
    lgb_models[(group, "LGBM_full")] = model_full
    pred_full = predictKw(model_full, X_valid, capacity_kw)[SEQ_LEN:]
    summary_full = summarizeAll(pred_full, actual_aligned, group)
    records.append(
        {
            "group": group,
            "model_name": "LGBM_full",
            "n_features": X_train.shape[1],
            "best_epoch": model_full.best_iteration_,
            **summary_full,
        }
    )
    # 전체 피처 LightGBM을 기준 모델과 최종 후보로 보관한다.

    sample = X_valid.sample(
        min(SHAP_SAMPLE_SIZE, len(X_valid)),
        random_state=SEED,
    )
    shap_values = FeatureSelector._shapArray(model_full, sample)
    shap_importance = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=X_train.columns,
    ).sort_values(ascending=False)
    shap_store[group] = (shap_values, sample, shap_importance)
    # 검증 표본의 평균 절대 SHAP 기여도를 내림차순으로 기록한다.

    selector = FeatureSelector(capacity_kw=capacity_kw)
    columns = selector.select(
        X_train,
        y_train,
        X_valid,
        y_valid,
        shap_importance,
        sample.index,
    )
    selected_features[group] = columns
    selection_reports[group] = selector.report_
    # 그룹마다 신호 분포가 다르므로 피처 선별도 별도로 수행한다.

    model_selected = trainLightGBM(
        X_train[columns],
        y_train,
        X_valid[columns],
        y_valid,
        capacity_kw,
    )
    lgb_models[(group, "LGBM_selected")] = model_selected
    pred_selected = predictKw(
        model_selected,
        X_valid[columns],
        capacity_kw,
    )[SEQ_LEN:]
    summary_selected = summarizeAll(pred_selected, actual_aligned, group)
    records.append(
        {
            "group": group,
            "model_name": "LGBM_selected",
            "n_features": len(columns),
            "best_epoch": model_selected.best_iteration_,
            **summary_selected,
        }
    )
    # 같은 설정으로 선별 피처 모델을 다시 학습해 선택 효과를 분리한다.

    persistence = BaselineModels.persistenceForecast(y_valid)[SEQ_LEN:]
    summary_persistence = summarizeAll(persistence, actual_aligned, group)
    records.append(
        {
            "group": group,
            "model_name": "Persistence_oracle_lag1",
            "n_features": 0,
            "best_epoch": np.nan,
            **summary_persistence,
        }
    )
    # 직전 실제 발전량을 쓰는 oracle은 비교선으로만 기록하고 제출 선택에서는 제외한다.

    print(
        f"[그룹{group}] 전체 {X_train.shape[1]} → 선별 {len(columns)}개 | "
        f"LGBM_full={summary_full['score']:.4f} | "
        f"LGBM_selected={summary_selected['score']:.4f}"
    )
    # 그룹별 피처 감소와 공식 총점 변화를 한 줄로 확인한다.

selection_path = f"{PREP_DIR}/baseline7_selected_features.csv"
selection_df = pd.concat(
    [
        pd.DataFrame(
            {
                "group": group,
                "feature": selected_features[group],
                "rank": range(1, len(selected_features[group]) + 1),
            }
        )
        for group in GROUPS
    ],
    ignore_index=True,
)
selection_df.to_csv(selection_path, index=False, encoding="utf-8-sig")
# 그룹별 SHAP 선별 피처와 순위를 LSTM 단계 전부터 재사용할 수 있게 저장한다.
print(f"✅ 피처 저장: {selection_path}")
# 긴 LSTM 실험을 나중에 실행해도 선별 결과를 먼저 보존한다.


In [ ]:
# 그룹별 상위 SHAP 피처와 선택 사유를 확인한다.
for group in GROUPS:
    print(f"=== KPX 그룹 {group}: SHAP 상위 15개 ===")
    display(selection_reports[group].head(15))
    # 중요도와 선택/제거 사유를 같은 표에서 읽는다.

    shap_values, sample, _ = shap_store[group]
    shap.summary_plot(shap_values, sample, max_display=15, show=False)
    plt.title(f"KPX 그룹 {group} — SHAP Summary 상위 15개")
    plt.xlabel("SHAP value (설비 이용률 기여도)")
    plt.tight_layout()
    plt.show()
    # 점의 색은 피처값, 가로 위치는 이용률 예측에 대한 기여 방향을 나타낸다.


---
## Step 3. 동일 선별 피처로 LSTM v1/v2/v3 공정 비교

baseline 6의 세 LSTM 표현 전략만 남기고, 입력은 Step 2의 그룹별 SHAP 선별 피처로 통일한다.

| 버전 | 표현 방식 |
|---|---|
| `v1` | 마지막 시각 hidden state |
| `v2` | LayerNorm + Dropout |
| `v3` | 24개 시각 Attention 가중합 |

결측 대체와 MinMax 스케일링은 학습 구간에만 적합한다. baseline 6과 달리 조기종료 기준도
MAE가 아니라 그룹별 공식 `0.5×(1-NMAE)+0.5×FICR`로 맞춘다.


In [ ]:
class OfficialScoreMonitor:
    """VersionedLSTMPipeline의 검증 monitor를 공식 그룹 총점으로 연결하는 클래스"""

    def __init__(self, metrics: EvaluationMetrics, group: int):
        self.metrics = metrics
        # 계단형 FICR을 그대로 계산하는 공식 지표 객체
        self.group = group
        # 이 LSTM이 담당하는 KPX 그룹 번호

    def score(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        capacity_kw: Optional[float] = None,
    ) -> float:
        """실제값·예측값으로 그룹 공식 총점을 반환하는 메서드"""
        del capacity_kw
        # 파이프라인 인터페이스 호환용 인자이며 그룹 용량은 metrics 내부 상수를 사용한다.
        return float(
            self.metrics.summarize(y_pred, y_true, self.group)["score"]
        )
        # 공식 그룹 총점이 클수록 좋은 조기종료 monitor를 반환한다.


MODEL_VERSIONS = ["v1", "v2", "v3"]
# baseline 6의 세 시퀀스 표현 전략
MODEL_PARAMS = {
    "epochs": 80,
    "warmup_epochs": 15,
    "loss_k": 40.0,
    "regression_weight": 0.75,
    "patience": 12,
    "batch_size": 256,
    # 64 → 256으로 에폭당 스텝을 1/4로 줄인다.
    "lr": 2e-3,
    # 배치가 4배가 되면 학습률을 sqrt(4)=2배로 올려 수렴 속도를 보정한다.
    "random_state": SEED,
}
# baseline 5의 안정화 설정으로 통일해 구조 이외의 차이를 줄인다.

lstm_artifacts: dict[tuple[int, str], dict] = {}
# 그룹·버전별 결측 변환기·스케일러·학습 파이프라인을 보관한다.

for group in GROUPS:
    data = datasets[group]
    columns = selected_features[group]
    X_train = data["X_train"][columns]
    X_valid = data["X_valid"][columns]
    y_train = data["y_train"]
    y_valid = data["y_valid"]
    # LightGBM과 동일한 SHAP 선별 피처·시간순 분할을 사용한다.

    variant = AllFeaturesVariant(random_state=SEED)
    train_frame = variant.fit_transform(X_train, y_train)
    valid_frame = variant.transform(X_valid)
    # 학습 중앙값으로 결측을 채우고 검증 컬럼 순서를 학습 계약에 맞춘다.
    scaler = MinMaxScaler().fit(train_frame)
    X_train_scaled = scaler.transform(train_frame)
    X_valid_scaled = scaler.transform(valid_frame)
    # MinMaxScaler는 학습 구간에만 fit해 검증 통계 누수를 차단한다.

    monitor = OfficialScoreMonitor(metrics, group)
    # 이 그룹의 NMAE·FICR 공식 총점을 조기종료 기준으로 사용한다.
    for version in MODEL_VERSIONS:
        pipeline = make_lstm_pipeline(
            version,
            capacity_kw=RATED_CAPACITY_KW[group],
            seq_len=SEQ_LEN,
            **MODEL_PARAMS,
        )
        pipeline.fit(
            X_train_scaled,
            y_train,
            X_val=X_valid_scaled,
            y_val=y_valid,
            metrics=monitor,
            group=group,
        )
        # 같은 입력·학습값에서 LSTM 표현 전략만 바꿔 공식 총점으로 조기종료한다.
        pred = pipeline.predict(X_valid_scaled, y_valid)
        actual = y_valid[SEQ_LEN:]
        # 슬라이딩 윈도우가 소비한 첫 24시간을 실제값에서도 동일하게 제외한다.
        summary = summarizeAll(pred, actual, group)
        model_name = f"LSTM_{version}_shap_selected"
        records.append(
            {
                "group": group,
                "model_name": model_name,
                "n_features": X_train_scaled.shape[1],
                "best_epoch": pipeline.best_epoch,
                **summary,
            }
        )
        lstm_artifacts[(group, model_name)] = {
            "variant": variant,
            "scaler": scaler,
            "pipeline": pipeline,
        }
        # 최종 후보 판단과 best_epoch 재사용을 위해 학습 객체를 보관한다.
        print(
            f"[그룹{group}][{version}] epoch={pipeline.best_epoch} | "
            f"NMAE={summary['nmae']:.4f} FICR={summary['ficr']:.4f} "
            f"총점={summary['score']:.4f}"
        )
        # 그룹·버전별 공식 지표를 공통 형식으로 출력한다.


---
## Step 4. 공식 총점 집계 + 그룹별 최적 모델 선택

결과 CSV는 baseline 5 HOWTO와 호환되는 핵심 컬럼을 유지하고 `best_epoch`를 추가한다.
최종 제출 모델은 Persistence를 제외하고 그룹별 검증 총점이 가장 높은 후보를 선택한다.


In [ ]:
def aggregateOfficialScore(result_df: pd.DataFrame) -> pd.DataFrame:
    """모델별 3그룹 평균 NMAE·FICR로 공식 총점을 집계하는 함수"""
    summary = result_df.groupby("model_name", as_index=False).agg(
        mean_nmae=("nmae", "mean"),
        mean_ficr=("ficr", "mean"),
        mean_mae=("mae", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_features=("n_features", "mean"),
        groups=("group", "nunique"),
    )
    # 모델별로 세 그룹 지표와 피처 수를 먼저 평균한다.
    summary["official_score"] = (
        0.5 * (1.0 - summary["mean_nmae"]) + 0.5 * summary["mean_ficr"]
    )
    # 대회 정의에 따라 평균 NMAE·FICR을 같은 비중으로 합친다.
    return summary.sort_values("official_score", ascending=False).reset_index(drop=True)
    # 제출 후보가 위로 오도록 공식 총점 내림차순으로 반환한다.


result_df = pd.DataFrame(records).drop_duplicates(
    subset=["group", "model_name"],
    keep="last",
)
# 셀을 다시 실행해도 같은 그룹·모델 행이 중복 누적되지 않게 마지막 기록만 남긴다.
summary_df = aggregateOfficialScore(result_df)
# 세 그룹을 모두 고려한 모델 계열별 공식 총점표를 만든다.

selectable = result_df[
    result_df["model_name"] != "Persistence_oracle_lag1"
].copy()
# 미래 실제값이 필요한 oracle 모델은 제출 후보에서 제외한다.
best_by_group = (
    selectable.sort_values(["group", "score"], ascending=[True, False])
    .groupby("group", as_index=False)
    .first()
)
# 각 그룹의 검증 총점 최고 제출 가능 모델을 한 행씩 선택한다.

display(summary_df)
print("=== 그룹별 최종 제출 모델 ===")
display(best_by_group[["group", "model_name", "n_features", "score"]])
# 전체 모델 순위와 실제 제출에 쓸 그룹별 후보를 함께 확인한다.

results_path = f"{ROOT}/BASELINE/baseline_7_results.csv"
summary_path = f"{ROOT}/BASELINE/baseline_7_summary.csv"
result_df.to_csv(results_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
# 그룹 원본 지표와 세 그룹 공식 집계를 리더보드 재사용 형식으로 저장한다.
print(f"✅ 결과 저장: {results_path}")
print(f"✅ 요약 저장: {summary_path}")
# 생성된 두 산출물 경로를 명시한다.


In [ ]:
# 모델별 세 그룹 공식 총점을 같은 축에서 비교한다.
plot_df = summary_df.sort_values("official_score")
colors = [
    "#B0B0B0" if "Persistence" in name else "#2E7D9A"
    for name in plot_df["model_name"]
]
# 제출 불가능한 Persistence만 회색, 나머지 후보는 파란색으로 표시한다.
plt.figure(figsize=(10, 0.55 * len(plot_df) + 2.0))
bars = plt.barh(plot_df["model_name"], plot_df["official_score"], color=colors)
for bar, value in zip(bars, plot_df["official_score"]):
    plt.text(
        value + 0.004,
        bar.get_y() + bar.get_height() / 2.0,
        f"{value:.4f}",
        va="center",
    )
    # 각 막대 오른쪽에 공식 총점 소수점 4자리를 표시한다.
plt.xlabel("공식 총점 0.5×(1-NMAE) + 0.5×FICR (클수록 좋음)")
plt.title("Baseline 7 — LightGBM·LSTM 통합 검증")
plt.xlim(0.0, max(plot_df["official_score"]) * 1.15)
plt.tight_layout()
plt.show()
# 모델 계열별 3그룹 평균 성능을 시각적으로 비교한다.


---
## Step 5. 전체 학습 + 평가 기간 예측 + 제출 검증

검증에서 정한 피처·모델·학습 반복 수를 고정하고 2022~2024년 전체 학습 데이터로 다시 적합한다.
LSTM은 학습 마지막 24시간 피처를 평가 입력 앞에 붙여 2025년 첫 시각부터 예측한다.
결측을 `0`이나 직전값으로 숨기지 않고 예측 길이·시각 정렬이 어긋나면 즉시 중단한다.


In [ ]:
def fitFinalLightGBM(
    X_full: pd.DataFrame,
    y_full: np.ndarray,
    capacity_kw: float,
    n_estimators: int,
) -> lgb.LGBMRegressor:
    """검증에서 정한 트리 수로 전체 학습 데이터에 LightGBM을 적합하는 함수"""
    params = {**LGB_PARAMS, "n_estimators": max(1, int(n_estimators))}
    # 검증 best_iteration_을 전체 학습의 고정 트리 수로 사용한다.
    model = lgb.LGBMRegressor(**params)
    model.fit(X_full, y_full / capacity_kw)
    # 전체 기간의 발전량(kWh)을 그룹 설비 이용률로 정규화해 학습한다.
    return model
    # 평가 기간 예측에 사용할 최종 트리 모델을 반환한다.


submission = loader["sample_submission"].copy()
# 원본 제출 양식을 복사해 행 순서와 필수 컬럼 순서를 보존한다.
submission["forecast_kst_dtm"] = pd.to_datetime(
    submission["forecast_kst_dtm"]
)
# 평가 피처와 같은 datetime 자료형으로 병합 키를 통일한다.

final_models: dict[int, object] = {}
# 그룹별 최종 학습 모델을 실행 중 검토할 수 있게 보관한다.
for group in GROUPS:
    target_col = f"kpx_group_{group}"
    model_name = best_by_group.loc[
        best_by_group["group"] == group,
        "model_name",
    ].iloc[0]
    train_df = datasets[group]["df"]
    test_df = prep_builder.load(group, "test")
    y_full = train_df[target_col].to_numpy(dtype=float)
    capacity_kw = RATED_CAPACITY_KW[group]
    # 이 그룹의 전체 실제 발전량(kWh)과 설비용량(kW)을 준비한다.

    if model_name.startswith("LGBM"):
        if model_name == "LGBM_full":
            columns = datasets[group]["feature_cols"]
        else:
            columns = selected_features[group]
        # 검증에서 선택된 전체/SHAP 피처 계약을 그대로 재사용한다.
        missing = [column for column in columns if column not in test_df.columns]
        if missing:
            raise KeyError(f"[그룹{group}] 평가 데이터에 없는 피처: {missing}")
        # 평가 컬럼 불일치를 조용히 재색인하지 않고 즉시 알린다.

        best_iteration = lgb_models[(group, model_name)].best_iteration_
        final_model = fitFinalLightGBM(
            train_df[columns],
            y_full,
            capacity_kw,
            best_iteration,
        )
        pred = predictKw(final_model, test_df[columns], capacity_kw)
        # 검증에서 정한 트리 수로 전체 기간을 학습하고 8,760시간을 예측한다.
    else:
        columns = selected_features[group]
        missing = [column for column in columns if column not in test_df.columns]
        if missing:
            raise KeyError(f"[그룹{group}] 평가 데이터에 없는 피처: {missing}")
        # LSTM도 LightGBM과 같은 SHAP 선별 피처 순서를 강제한다.

        variant = AllFeaturesVariant(random_state=SEED)
        full_frame = variant.fit_transform(train_df[columns], y_full)
        test_frame = variant.transform(test_df[columns])
        # 전체 학습 중앙값으로 평가 결측을 채우고 컬럼 순서를 고정한다.
        scaler = MinMaxScaler().fit(full_frame)
        X_full_scaled = scaler.transform(full_frame)
        X_test_scaled = scaler.transform(test_frame)
        # 최종 스케일러도 라벨이 있는 전체 학습 기간에만 fit한다.

        version = model_name.split("_")[1]
        best_epoch = int(lstm_artifacts[(group, model_name)]["pipeline"].best_epoch)
        final_params = {
            **MODEL_PARAMS,
            "epochs": max(1, best_epoch),
            "patience": max(2, best_epoch + 1),
        }
        # 검증에서 고른 epoch 수만큼 전체 학습하고 조기종료는 발동하지 않게 한다.
        final_model = make_lstm_pipeline(
            version,
            capacity_kw=capacity_kw,
            seq_len=SEQ_LEN,
            **final_params,
        )
        final_model.fit(X_full_scaled, y_full)
        # 2022~2024년 전체 시퀀스로 최종 LSTM 가중치를 학습한다.
        X_with_context = np.vstack(
            [X_full_scaled[-SEQ_LEN:], X_test_scaled]
        )
        # 학습 마지막 24시간을 붙여 2025년 첫 시각의 과거 문맥을 제공한다.
        pred = final_model.predict(X_with_context)
        # 문맥 24행이 소비되므로 반환 길이는 평가 8,760행과 정확히 같아야 한다.

    if len(pred) != len(test_df):
        raise ValueError(
            f"[그룹{group}] 예측 길이 불일치: 예측 {len(pred)}, 평가 {len(test_df)}"
        )
        # 첫 24시간 누락을 0으로 숨기던 이전 제출 버그의 재발을 차단한다.
    final_models[group] = final_model
    # 그룹별 최종 모델을 보관한다.

    pred_series = pd.Series(
        pred,
        index=pd.to_datetime(test_df["forecast_kst_dtm"]),
    )
    submission[target_col] = pred_series.reindex(
        submission["forecast_kst_dtm"]
    ).to_numpy()
    # 예측 시각 인덱스로 sample_submission의 정확한 행 순서에 맞춰 넣는다.
    print(
        f"✅ [그룹{group}] {model_name} | 피처 {len(columns)}개 | "
        f"예측 {len(pred):,}시간"
    )
    # 그룹별 선택 모델·피처 수·예측 길이를 실행 로그에 남긴다.


In [ ]:
# 제출 스키마·결측·물리 범위·첫 24시간 예측을 검증한다.
sample = loader["sample_submission"]
assert list(submission.columns) == list(sample.columns), "제출 컬럼 순서 불일치!"
assert len(submission) == len(sample) == 8_760, "제출 행 수가 8,760이 아닙니다!"
assert submission.isna().sum().sum() == 0, "제출 파일에 NaN이 있습니다!"
# 결측을 ffill/0으로 숨기지 않고 시각 정렬 실패를 그대로 중단한다.
for group in GROUPS:
    column = f"kpx_group_{group}"
    assert (submission[column] >= 0.0).all(), f"{column}에 음수 예측값이 있습니다!"
    assert (
        submission[column] <= RATED_CAPACITY_KW[group]
    ).all(), f"{column}에 설비용량 초과 예측값이 있습니다!"
    assert submission[column].nunique() > 1, f"{column} 예측이 상수입니다!"
    # 발전량은 0~그룹 설비용량(kWh/h) 범위이며 전체 기간이 상수로 무너지면 안 된다.

out_path = f"{ROOT}/submission_baseline7.csv"
submission.to_csv(out_path, index=False, encoding="utf-8-sig")
# 대회 제출 호환을 위해 UTF-8 BOM·인덱스 제외 형식으로 저장한다.
print(f"스키마 검증 통과 ✅")
print(f"제출 파일 저장 완료: {out_path}")
# 검증 성공과 최종 파일 경로를 명시한다.
display(submission.head())
# 2025년 첫 시각부터 세 그룹 예측이 채워졌는지 확인한다.


In [ ]:
# 최종 제출 예측이 입력에 반응하는지 그룹별 분포를 확인한다.
for group in GROUPS:
    column = f"kpx_group_{group}"
    series = submission[column]
    print(
        f"[{column}] 고유값 {series.nunique():,}개 | "
        f"평균 {series.mean():,.0f} | 최소 {series.min():,.0f} | "
        f"최대 {series.max():,.0f} kWh"
    )
    # 고유값이 충분하고 0~설비용량 범위면 상수 예측 붕괴 가능성이 낮다.

submission.set_index("forecast_kst_dtm")[
    [f"kpx_group_{group}" for group in GROUPS]
].head(24 * 14).plot(
    figsize=(12, 4),
    title="Baseline 7 제출 예측 — 2025년 첫 2주",
)
plt.ylabel("예측 발전량 (kWh)")
plt.xlabel("예보 대상 시각")
plt.tight_layout()
plt.show()
# 첫 2주 시간 변화와 그룹별 차이를 마지막으로 시각 검토한다.


---
## 정리

| 통합 항목 | Baseline 7 적용 |
|---|---|
| IDW 버그 차단 | 그룹 문자열 정규화 + LDAPS/GFS 대표 풍속 0 행렬 검사 |
| baseline 5 HOWTO | 예보 변화량 피처, LightGBM, SHAP/null importance, 상관 중복 제거 |
| baseline 6 | LSTM v1/v2/v3를 같은 SHAP 피처·학습 조건에서 비교 |
| 공정 비교 | 동일한 마지막 20%, 동일한 첫 24시간 제외, 공식 총점 조기종료 |
| 제출 | 그룹별 최고 제출 가능 모델 전체 재학습, LSTM 과거 24시간 문맥 연결 |
| 산출물 | `baseline7_selected_features.csv`, `baseline_7_results.csv`, `baseline_7_summary.csv`, `submission_baseline7.csv` |

실제 채택 모델과 점수는 Step 4의 실행 결과를 기준으로 판단한다. 저장된 baseline 5/6의 과거 출력은
수정 전 코드 또는 다른 전처리 조건일 수 있으므로 baseline 7의 공통 실행 결과와 직접 섞지 않는다.
